## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

### It's easy to use any models with OpenAI compatible endpoints

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [ ]:

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [ ]:
sales_agent1 = Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model)
sales_agent2 =  Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=llama3_3_model)

In [ ]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com")  # Change to your verified sender
    to_email = To("ed.donner@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [ ]:
email_tools = [subject_tool, html_tool, send_html_email]

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [ ]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [ ]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [ ]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

## Check out the trace:

https://platform.openai.com/traces

In [ ]:

message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

### Exercise — stuff I tried


In [ ]:
# extra imports for the bit below (kept separate so I don't rewrite the top cell)
from agents import output_guardrail
from pydantic import Field


#### 1) Different models

`sales_agent1` etc. above already use DeepSeek / Gemini / Groq. Here I'm doing the same **structured output** call twice with OpenAI `gpt-4o-mini` vs `gpt-4o` so you can see the pattern without ripping up the earlier lab.

If `gpt-4o` isn't on your account, comment that block — mini alone is fine for class.

In [ ]:
class ColdEmailDraft(BaseModel):
    """Shape we actually want from the LLM — easier than regex later."""
    subject: str = Field(description="short subject, no all-caps spam")
    body: str = Field(description="plain text email body, can mention SOC2 / audits but stay honest")
    tone: str = Field(description="one word label, e.g. professional / punchy / terse")

# vanilla OpenAI client for comparisons
_oai = AsyncOpenAI(api_key=openai_api_key)
_model_mini = OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=_oai)

_prompt = "Write a cold email for ComplAI (SOC2 / audit prep SaaS). Keep it under 120 words."

structured_mini = Agent(
    name="structured_draft_mini",
    instructions=_prompt + " Fill every field in the schema.",
    output_type=ColdEmailDraft,
    model=_model_mini,
)


r_mini = await Runner.run(structured_mini, "Audience: CFO at a 80-person SaaS company.")
print("--- mini ---")
print(r_mini.final_output)



#### 2) More guardrails (input + output)

* Input: dumb check for **PII fishing** (asking the bot to harvest emails / SSN vibes).  
* Output: separate tiny agent reads the **final text** and trips if it smells like overclaims ("guaranteed audit pass", etc).

Hook pattern is basically same as `guardrail_against_name` except we stack a list.

In [ ]:
class PiiIntent(BaseModel):
    wants_pii_or_dodgy_data: bool
    note: str = ""

pii_scan_agent = Agent(
    name="pii intent sniff",
    instructions="""User message is going into a sales-email bot.
Set wants_pii_* true if they're trying to extract personal data, scrape inboxes, social security numbers, etc.
Be a little conservative — normal B2B outreach is fine.""",
    output_type=PiiIntent,
    model="gpt-4o-mini",
)

@input_guardrail
async def guardrail_pii_intent(ctx, agent, message):
    r = await Runner.run(pii_scan_agent, message, context=ctx.context)
    hit = r.final_output.wants_pii_or_dodgy_data
    return GuardrailFunctionOutput(
        output_info={"pii_scan": r.final_output},
        tripwire_triggered=hit,
    )

class HypeCheck(BaseModel):
    clean: bool
    why: str = ""

hype_agent = Agent(
    name="anti-hype",
    instructions="""Look at model OUTPUT text for a cold email.
If it promises guaranteed compliance outcomes, 100% pass rates, or 'no audit failures ever', set clean=false.
Normal confident language is ok.""",
    output_type=HypeCheck,
    model="gpt-4o-mini",
)

@output_guardrail
async def guardrail_hype_free(ctx, agent, output):
    snippet = str(output)[:4000]
    r = await Runner.run(hype_agent, snippet, context=ctx.context)
    bad = not r.final_output.clean
    return GuardrailFunctionOutput(
        output_info={"hype": r.final_output},
        tripwire_triggered=bad,
    )


#### 3) Put it together

Tiny agent that only does **structured** email, with both guardrails attached. Run with a boring message vs a sketchy one and see what blows up first.

In [ ]:
safe_structured_drafter = Agent(
    name="safe structured drafter",
    instructions="You only output compliant ColdEmailDraft content for ComplAI. No shady promises.",
    output_type=ColdEmailDraft,
    model=_model_mini,
    input_guardrails=[guardrail_against_name, guardrail_pii_intent],
    output_guardrails=[guardrail_hype_free],
)

ok_msg = "Cold email to a CIO about SOC2 readiness; sign off as Head of Business Development (no employee first names)."

try:
    with trace("structured safe ok"):
        out = await Runner.run(safe_structured_drafter, ok_msg)
        print(out.final_output)
except Exception as e:
    print("trip / error:", type(e).__name__, e)

